# core

> Sanskrit recitation from text with Vāgdhenu: a chant voice for verse, a prose voice for gadya, and line timings.

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

This module loads Vāgdhenu and renders Sanskrit text with it. Vāgdhenu ([code](https://github.com/prathoshap/vagdhenu), [weights](https://huggingface.co/prathoshap/vagdhenu), Apache-2.0) is a fine-tune of IndicF5, a text-to-speech model, with a BigVGAN-v2 vocoder, trained on one speaker's chanting. Its source code, BigVGAN's and IndicF5's (the `f5_tts` package, which is not the `f5-tts` on PyPI) are not pip packages. So on first use `fetch_src` downloads each of them at a fixed commit, listed in `SRCS`, into the cache folder, and `setup_paths` puts them on `sys.path`. The weights come from the Hugging Face Hub. `best_device` picks CUDA if it is there, then Apple's MPS, then the CPU.

In [ ]:
#| export
import os, re, sys, io, tarfile, types, subprocess, unicodedata, urllib.request, numpy as np
from pathlib import Path
from dataclasses import dataclass, field
from fastcore.utils import *
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')

In [ ]:
#| export
SRCS = dict(vagdhenu=('prathoshap/vagdhenu', 'c18927a8a77775753e32248177a5a6bea1019a12', 'src'),
            bigvgan=('NVIDIA/BigVGAN', '7d2b454564a6c7d014227f635b7423881f14bdac', ''),
            indicf5=('ai4bharat/IndicF5', '13f7c4d627cc10111aea8fe9c0039462cacacdc7', ''))
HF = 'prathoshap/vagdhenu'
VOICES = dict(steer='voice_steer_ema_2026-06-17.pt', armA='voice_armA_ema_2026-06-11.pt')
VOC, SR = 'voc_bigvgan_EMA_2026-06-11.pth', 24000
cache = Path(os.environ.get('VACHAKA_CACHE', '~/.cache/vachaka')).expanduser()

In [ ]:
#| export
def fetch_src(name):
    "The path to the source code of `name` at its fixed commit, downloaded on first use."
    repo, sha, sub = SRCS[name]
    d = cache/f'{name}-{sha[:10]}'
    if not d.exists():
        data = urllib.request.urlopen(f'https://codeload.github.com/{repo}/tar.gz/{sha}').read()
        with tarfile.open(fileobj=io.BytesIO(data)) as t: t.extractall(d, filter='data')
    return next(d.iterdir())/sub

In [ ]:
#| export
def setup_paths():
    "Put Vāgdhenu's `src`, BigVGAN and IndicF5 on `sys.path`."
    for n in SRCS:
        p = str(fetch_src(n))
        if p not in sys.path: sys.path.insert(0, p)

In [ ]:
#| export
def best_device():
    "The best available torch device: cuda, then mps, then cpu."
    import torch
    if torch.cuda.is_available(): return 'cuda'
    return 'mps' if torch.backends.mps.is_available() else 'cpu'

## Text

The model reads plain Devanagari, so `clean(s)` removes Vedic accent marks and digits, and reduces each run of daṇḍas (`।` and `॥`, with the verse number between them) to a single daṇḍa, keeping `॥` where there was one. Vāgdhenu has no svaras (Vedic pitch accents) and chants in the classical, laukika, style.

`n_syll(s)` counts the akṣaras, the syllables of Devanagari text: each independent vowel and each consonant not followed by a virāma. `chunk_prose(text, max_syll)` splits prose into pieces Vāgdhenu can render: first at daṇḍas, then between words, so that no piece has more than `max_syll` (24) akṣaras.

In [ ]:
#| export
_ACC = re.compile('[\u0951-\u0954\u1CD0-\u1CFF\uA8E0-\uA8FF]')
def clean(s):
    "Plain Devanagari for the model, without accents, digits or verse numbers between daṇḍas."
    s = _ACC.sub('', s)
    s = re.sub('[0-9\u0966-\u096F]+', '', s)
    s = re.sub('[।॥|]+(\\s*[।॥|]+)*', lambda m: '॥' if '॥' in m[0] else '।', s)
    return re.sub('\\s+', ' ', s).strip()

In [ ]:
test_eq(clean('अग्निमी॑ळे पु॒रोहि॑तं ॥ १ ॥'), 'अग्निमीळे पुरोहितं ॥')
test_eq(clean('धर्मक्षेत्रे कुरुक्षेत्रे ।'), 'धर्मक्षेत्रे कुरुक्षेत्रे ।')

In [ ]:
#| export
def n_syll(s):
    "The number of akṣaras in Devanagari `s`: independent vowels and consonants without a virāma."
    return sum(1 for i,c in enumerate(s) if '\u0905'<=c<='\u0914' or ('\u0915'<=c<='\u0939' and s[i+1:i+2]!='्'))

In [ ]:
test_eq(n_syll('कर्मण्येवाधिकारस्ते'), 8)

In [ ]:
#| export
def chunk_prose(text, max_syll=24):
    "Split prose at daṇḍas, then between words, into pieces of at most `max_syll` akṣaras."
    out = []
    for sent in re.split('[।॥|]', clean(text)):
        cur, cs = [], 0
        for w in sent.split():
            ws = n_syll(w)
            if cur and cs+ws > max_syll: out.append(' '.join(cur)); cur, cs = [], 0
            cur.append(w); cs += ws
        if cur: out.append(' '.join(cur))
    return out

In [ ]:
p = chunk_prose('अथातो धर्मजिज्ञासा । चोदनालक्षणोऽर्थो धर्मः । तस्य निमित्तपरीष्टिः ॥')
test_eq(p, ['अथातो धर्मजिज्ञासा', 'चोदनालक्षणोऽर्थो धर्मः', 'तस्य निमित्तपरीष्टिः'])
assert all(n_syll(c) <= 24 for c in chunk_prose('क '*80))

## Sound

The raw render can sound thin, so `polish(src, dst, fx)` applies a named sound treatment to an audio file with ffmpeg. The treatments in `FX` were chosen by ear for listeners, not sound engineers. Each ends by normalising loudness to -16 LUFS, so every recording plays at the same volume, and none moves any sound in time: the echo adds decay after a syllable, never delay before it, so line and word timings stay valid. `warm`, the default, adds a little warmth below 250 Hz, softens hissing s sounds and adds the short echo of a small room. `phone` brings the words forward for small speakers. `plain` only normalises loudness.

In [ ]:
#| export
LN = 'loudnorm=I=-16:TP=-1.5:LRA=11'
FX = dict(plain=LN,
          warm=f'highpass=f=70,equalizer=f=180:t=q:w=1:g=2.5,deesser=i=0.4,aecho=0.8:0.6:35|70:0.22|0.12,{LN}',
          phone=f'highpass=f=90,acompressor=threshold=-20dB:ratio=2.5:attack=10:release=150,equalizer=f=3000:t=q:w=1.2:g=3,{LN}')

def polish(src, dst, fx='warm', kbps=96):
    "Apply the sound treatment `FX[fx]` to audio file `src` and write it to `dst`."
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(src), '-af', FX[fx], '-b:a', f'{kbps}k', str(dst)], check=True)
    return Path(dst)

In [ ]:
test_eq(sorted(FX), ['phone', 'plain', 'warm'])
assert all(v.endswith(LN) for v in FX.values())

## Recitation

A `Recitation` is what every render returns: the audio as a numpy array, its sample rate `sr`, one `(start_ms, end_ms)` span per piece in `spans`, the text of each piece in `texts`, and, once Su-śrotā has checked it, the character error rate `cer`. A piece is a line given to `Chanter.chant` or a chunk made by `Chanter.read`. `ms` is the length in milliseconds. `save` writes a wav file, or an mp3 through ffmpeg when the path ends in `.mp3`, optionally shifted `pitch` semitones and with an `FX` treatment; the timings stay valid. `concat`, or `+`, joins recitations with a silence between them and moves the later spans along to match.

In [ ]:
#| export
@dataclass
class Recitation:
    audio: np.ndarray
    spans: list = field(default_factory=list)
    texts: list = field(default_factory=list)
    sr: int = SR
    cer: float = None
    @property
    def ms(self): return round(len(self.audio)*1000/self.sr)
    def __add__(self, o): return concat([self, o])
    def save(self, path, kbps=96, fx=None, pitch=0):
        "Write a wav file, or an mp3 when `path` ends in .mp3, shifted `pitch` semitones and with the sound treatment `fx`."
        import soundfile as sf
        path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
        au = self.audio
        if pitch:
            import librosa
            au = librosa.effects.pitch_shift(np.asarray(au, np.float32), sr=self.sr, n_steps=pitch)
        if path.suffix != '.mp3' and not fx: sf.write(path, au, self.sr); return path
        buf = io.BytesIO(); sf.write(buf, au, self.sr, format='WAV')
        af = ['-af', FX[fx]] if fx else []
        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-f', 'wav', '-i', '-', *af, *(['-b:a', f'{kbps}k'] if path.suffix == '.mp3' else []), str(path)], input=buf.getvalue(), check=True)
        return path

In [ ]:
#| export
def concat(recs, gap=0.8):
    "Join recitations with `gap` seconds of silence between them, moving their spans to match."
    recs = L(recs); sr = recs[0].sr; sil = np.zeros(int(gap*sr), np.float32)
    aus, spans, texts, pos = [], [], [], 0
    for i,r in enumerate(recs):
        if i: aus.append(sil); pos += len(sil)
        off = round(pos*1000/sr)
        spans += [(s+off, e+off) for s,e in r.spans]; texts += r.texts
        aus.append(r.audio); pos += len(r.audio)
    return Recitation(np.concatenate(aus), spans, texts, sr)

In [ ]:
a = Recitation(np.zeros(SR, np.float32), [(0, 900)], ['क'])
b = a + a
test_eq(b.ms, 2800)
test_eq(b.spans, [(0, 900), (1800, 2700)])

## Chanter

`Chanter()` loads Vāgdhenu's network, the vocoder and the bank of reference recordings once, and then renders as often as you like. Vāgdhenu sings a text in the style of a reference recording: the reference sets the pace and melody, and the timbre is always the fine-tuned voice. The bank holds one reference for each metre Vāgdhenu supports, plus `gadya` references for prose.

- `chant(lines)` sings one verse given as a list of its lines, or as a string that it splits at newlines and daṇḍas. It finds the metre with [ganapati](https://github.com/vedicreader/ganapati) through `meter`, renders against that metre's reference, and uses vasantatilakā when nothing is detected. Pass `meter=` to choose one. The result has one span per line.
- `read(text)` reads prose, split by `chunk_prose`, against the `gadya` reference by default, or against another reference given as `ref`. The result has one span per chunk.
- `render(pieces, ref)` is what both call: it renders already cleaned pieces against one reference.
- `meters()` lists the metres in the bank, and `bank_ref(name)` finds a metre's reference under any spelling (IAST, plain ASCII or the recording's file name), or returns None.

`pace` stretches every syllable; the default of 1.15 is unhurried, for chanting along. `seed` picks the random seed for a render; a different seed gives a slightly different performance. `voice` chooses between the two fine-tuned voices in `VOICES`. On Apple silicon the main network runs on MPS and BigVGAN on the CPU, because MPS has no transposed convolution with more than 65,536 output channels.

In [ ]:
#| export
def _stitch(self, segs, gaps, fric=False, halant=False):
    "Render-core's stitch, also recording each gated piece's span in samples; each piece keeps `tail` s past its voice, faded over `fade` s, and the last is followed by `pad` s of silence."
    from render_core import gate
    last, out, self.spans, pos = len(segs)-1, [], [], 0
    for i,s in enumerate(segs):
        g = gate(s, fric=fric and i==0, halant=halant and i==last, keep=self.tail, fout=self.fade)
        self.spans.append((pos, pos+len(g))); pos += len(g); out.append(g)
        if i < last: gp = gaps[i] if i < len(gaps) else gaps[-1]; out.append(gp); pos += len(gp)
    out.append(np.zeros(int(self.pad*SR), np.float32))
    return np.concatenate(out)

In [ ]:
#| export
def _cpu_vocoder(r):
    "Move BigVGAN to the CPU."
    import torch
    r.g = r.g.to('cpu')
    def bv(self, mel):
        m = torch.from_numpy(mel)
        with torch.no_grad():
            if m.dim() == 3 and m.shape[1] != 100 and m.shape[2] == 100: m = m.transpose(1, 2)
            return self.g(m).squeeze().numpy().astype(np.float32)
    r._bvgan = types.MethodType(bv, r)

In [ ]:
#| export
class Chanter:
    "Renders Sanskrit verse and prose to audio with Vāgdhenu, with a time span per line."
    def __init__(self, voice='steer', device=None, nfe=64, speed=0.9, gap=0.55, prose_gap=0.3, tail=0.18, fade=0.12, pad=0.5, pace=1.15):
        from huggingface_hub import hf_hub_download
        setup_paths()
        from render_core import Renderer
        device = ifnone(device, best_device())
        store_attr('voice,device,gap,prose_gap,pace')
        dl = partial(hf_hub_download, HF)
        bank = fetch_src('vagdhenu')/'reference_bank'/'bank.json'
        self.r = Renderer(dl(VOICES[voice]), dl(VOC), str(bank), device=device, vocab_file=dl('vocab.txt'), nfe=nfe, speed=speed, gap=gap)
        self.r._stitch = types.MethodType(_stitch, self.r)
        self.r.tail, self.r.fade, self.r.pad = tail, fade, pad   # Vāgdhenu's gate kept 60 ms past the voice and faded over 40: words ended abruptly
        if device == 'mps': _cpu_vocoder(self.r)

    def meters(self): return self.r.meters()
    def meter(self, lines):
        "The metre of a verse given as its lines, found by ganapati, or ''."
        from ganapati.metre import detect_meter
        m = detect_meter('\n'.join(lines))
        return m.name.value if m and m.name else ''
    def bank_ref(self, name):
        "The bank's reference for metre `name`, in IAST, ASCII or file-name spelling, or None."
        k = ''.join(c for c in unicodedata.normalize('NFKD', name or '') if c.isascii() and c.isalnum()).lower()
        return next((o for o in ((name or '').lower(), k) if o in self.r._lut), None)

    def render(self, pieces, ref, seed=60, gap=None):
        "Render cleaned `pieces` against the reference `ref`, a metre or gadya, with `gap` seconds between them."
        self.r.gap = ifnone(gap, self.gap)
        sps = self.r._get_ref(ref)[2] * self.pace if self.pace != 1 else None   # pace: the reference's seconds a syllable, times this (1.15: unhurried, to chant along)
        sr, au = self.r.render_one(pieces, ref, seed=seed, sps=sps)
        return Recitation(au, [(round(s*1000/sr), round(e*1000/sr)) for s,e in self.r.spans], pieces, sr)

    def chant(self, lines, meter=None, seed=60):
        "Chant one verse given as its lines, or as a string split at newlines and daṇḍas, with one span per line."
        if isinstance(lines, str): lines = re.split('[\n।॥|]', lines)
        lines = [o for o in (clean(l).strip(' ।॥') for l in lines) if o]
        return self.render(lines, ifnone(meter, self.meter(lines)) or 'vasantatilaka', seed)

    def read(self, text, ref='gadya', seed=60, max_syll=24):
        "Read prose against the reference `ref`, gadya by default, with one span per chunk."
        return self.render(chunk_prose(text, max_syll), ref, seed, self.prose_gap)

Rendering needs the weights, about 1.5 GB downloaded on first use, and takes several times real time on Apple silicon, so these cells do not run in the tests.

In [ ]:
#| eval: false
ch = Chanter()
r = ch.chant(['वसुदेवसुतं देवं कंसचाणूरमर्दनम्', 'देवकीपरमानन्दं कृष्णं वन्दे जगद्गुरुम्'])
r.spans, r.save('/tmp/krishna.mp3')

In [ ]:
#| eval: false
ch.read('अथातो धर्मजिज्ञासा । चोदनालक्षणोऽर्थो धर्मः ।').save('/tmp/mimamsa.mp3')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()